<a href="https://colab.research.google.com/github/LarshVakil/Semantic-Search-Engine/blob/main/Poker_Calculator_Hands.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import random as rd
from collections import Counter

class PokerMatrixSim:
    def __init__(self):
        self.ranks = list(range(2, 15))
        self.suits = list(range(1, 5))
        self.deck = [(r, s) for r in self.ranks for s in self.suits]
        self.hand_types = ["High Card", "Pair", "Two Pair", "3-of-a-Kind",
                          "Straight", "Flush", "Full House", "4-of-a-Kind", "Str. Flush"]

    def get_hand_str(self, hand):
        h = sorted(hand, key=lambda x: x[0], reverse=True)
        r1, r2 = h[0][0], h[1][0]
        rank_map = {14:'A', 13:'K', 12:'Q', 11:'J', 10:'T'}
        n1 = rank_map.get(r1, str(r1))
        n2 = rank_map.get(r2, str(r2))

        if r1 == r2: return n1 + n2
        suffix = 's' if h[0][1] == h[1][1] else 'o'
        return n1 + n2 + suffix

    def evaluate(self, hand, board):
        all_c = hand + board
        r = sorted([c[0] for c in all_c], reverse=True)
        s = [c[1] for c in all_c]
        rc = Counter(r).most_common()
        sc = Counter(s).most_common()


        is_f = sc[0][1] >= 5
        f_suit = sc[0][0] if is_f else None
        u_r = sorted(list(set(r)), reverse=True)
        st_h = None
        for i in range(len(u_r)-4):
            if u_r[i] - u_r[i+4] == 4: st_h = u_r[i]; break
        if not st_h and {14,2,3,4,5}.issubset(set(u_r)): st_h = 5

        if is_f and st_h:
            fc = sorted([c[0] for c in all_c if c[1] == f_suit], reverse=True)
            for i in range(len(fc)-4):
                if fc[i] - fc[i+4] == 4: return 8, 800+fc[i]
            if {14,2,3,4,5}.issubset(set(fc)): return 8, 805
        if rc[0][1] == 4: return 7, 700+rc[0][0]
        if rc[0][1] == 3 and len(rc)>1 and rc[1][1] >= 2: return 6, 600+rc[0][0]
        if is_f: return 5, 500 + max([c[0] for c in all_c if c[1] == f_suit])
        if st_h: return 4, 400 + st_h
        if rc[0][1] == 3: return 3, 300+rc[0][0]
        if rc[0][1] == 2 and len(rc)>1 and rc[1][1] == 2: return 2, 200+rc[0][0]
        if rc[0][1] == 2: return 1, 100+rc[0][0]
        return 0, r[0]

    def run(self, iterations=1000000):
        stats = {}

        for _ in range(iterations):
            rd.shuffle(self.deck)
            hA, hB, board = self.deck[0:2], self.deck[2:4], self.deck[4:9]

            typeA, scoreA = self.evaluate(hA, board)
            typeB, scoreB = self.evaluate(hB, board)

            strA, strB = self.get_hand_str(hA), self.get_hand_str(hB)

            for s in [strA, strB]:
                if s not in stats: stats[s] = np.zeros(11)

            stats[strA][0] += 1
            stats[strB][0] += 1

            if scoreA > scoreB:
                stats[strA][1] += 1
                stats[strA][typeA + 2] += 1
            elif scoreB > scoreA:
                stats[strB][1] += 1
                stats[strB][typeB + 2] += 1

        return stats

In [ ]:
sim = PokerMatrixSim()
raw_data = sim.run(20000000)


cols = ['Played', 'Wins'] + sim.hand_types
df = pd.DataFrame.from_dict(raw_data, orient='index', columns=cols)


df['Winrate %'] = (df['Wins'] / df['Played'] * 100).round(2)
for h_type in sim.hand_types:

    df[f'{h_type} %'] = (df[h_type] / df['Wins'] * 100).round(1)


final_table = df[['Winrate %'] + [f'{t} %' for t in sim.hand_types]].sort_values('Winrate %', ascending=False)
print(final_table)

     Winrate %  High Card %  Pair %  Two Pair %  3-of-a-Kind %  Straight %  \
AA       84.40          0.0    34.1        40.0           12.8         0.8   
KK       79.34          0.0    34.5        38.4           13.6         0.9   
QQ       75.51          0.0    34.4        37.3           14.0         1.5   
JJ       72.02          0.0    34.0        35.9           14.5         2.0   
TT       68.78          0.0    33.5        34.8           15.0         2.5   
..         ...          ...     ...         ...            ...         ...   
42o      29.68          0.0    29.1        36.8            9.1        14.4   
82o      29.66          0.0    34.0        39.2            9.1         7.1   
62o      29.45          0.0    31.6        38.3            8.9        10.6   
32o      29.21          0.0    28.4        38.2            9.2        13.6   
72o      28.86          0.0    33.5        40.0            9.3         6.4   

     Flush %  Full House %  4-of-a-Kind %  Str. Flush %  
AA   